# 🛰️ GeoAI EuroSAT 数据探索

探索 EuroSAT 遥感图像数据集，可视化各类别样本分布。

In [ ]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'scripts'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from dataset import EuroSATDataset, CLASS_NAMES, get_val_transforms
print('✅ 导入成功')

In [ ]:
# 加载数据集
data_root = ROOT / 'data' / 'processed' / 'EuroSAT'
try:
    train_ds = EuroSATDataset(data_root, split='train')
    print(f'训练集样本数: {len(train_ds)}')
    print(f'类别分布: {train_ds.get_class_distribution()}')
except FileNotFoundError as e:
    print(f'⚠ {e}\n请先运行: python scripts/download_dataset.py')

In [ ]:
# 可视化类别分布
dist = train_ds.get_class_distribution()
fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.cm.Set3(np.linspace(0, 1, len(dist)))
bars = ax.bar(dist.keys(), dist.values(), color=colors)
ax.set_title('EuroSAT 训练集类别分布', fontsize=14, fontweight='bold')
ax.set_xlabel('地物类别'); ax.set_ylabel('样本数量')
plt.xticks(rotation=30, ha='right')
for bar, val in zip(bars, dist.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
            str(val), ha='center', va='bottom', fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# 可视化每类样例图像
from PIL import Image
import random

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for i, cls in enumerate(CLASS_NAMES):
    cls_samples = [(p, l) for p, l in train_ds.samples if CLASS_NAMES[l] == cls]
    if cls_samples:
        img_path, _ = random.choice(cls_samples)
        img = Image.open(img_path).convert('RGB').resize((64, 64))
        axes[i].imshow(img)
    axes[i].set_title(cls, fontsize=9)
    axes[i].axis('off')

fig.suptitle('EuroSAT 各类别样例图像', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# 数据增强可视化
import torch
from dataset import get_train_transforms
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406])
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225])

def denorm(t):
    return (t * IMAGENET_STD[:,None,None] + IMAGENET_MEAN[:,None,None]).clamp(0,1)

tfm = get_train_transforms(224)
img_path, label = train_ds.samples[0]
orig = Image.open(img_path).convert('RGB')

fig, axes = plt.subplots(1, 6, figsize=(14, 3))
axes[0].imshow(orig.resize((224,224))); axes[0].set_title('原图'); axes[0].axis('off')
for j in range(1, 6):
    aug = denorm(tfm(orig)).permute(1,2,0).numpy()
    axes[j].imshow(aug); axes[j].set_title(f'增强{j}'); axes[j].axis('off')

fig.suptitle(f'数据增强示例 ({CLASS_NAMES[label]})', fontsize=12)
plt.tight_layout(); plt.show()